In [ ]:
import torch
import torch.nn as nn
import math
from torch.nn import functional as F
import argparse

# data & training params
BATCH_SIZE = 32
BLOCK_SIZE = 256
MAX_ITERS = 10000
EVAL_INTERVAL = 250
LEARNING_RATE = 3e-4
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
EVAL_ITERS = 200
DATA_FILE_PATH = 'clean_truths.txt'

# architecture params ; comparable to gpt-2
N_EMBD = 768
N_HEAD = 12
N_LAYER = 12
DROPOUT = 0.2

# rate schedular params
WARMUP_ITERS = 200
LR_DECAY_ITERS = 10000
MIN_LR = 3e-5

# gradient clipping
GRAD_CLIP = 1.0

# pytorch stuff
torch.manual_seed(1337)
torch.set_float32_matmul_precision('high')

# loading training data
try:
    with open(DATA_FILE_PATH, 'r', encoding='utf-8') as f:
        text = f.read()
except FileNotFoundError:
    print(f"Error: The file '{DATA_FILE_PATH}' was not found.")
    print("Please make sure the dataset file is in the same directory or provide the correct path.")
    exit()

import tiktoken
enc = tiktoken.get_encoding("gpt2")
vocab_size = enc.n_vocab
encode = lambda s: enc.encode(s, allowed_special={"<|endoftext|>"})
decode = lambda l: enc.decode(l)

# helpers

def get_batch(split, train_data, val_data):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE,))
    x = torch.stack([data[i:i + BLOCK_SIZE] for i in ix])
    y = torch.stack([data[i + 1:i + BLOCK_SIZE + 1] for i in ix])
    x, y = x.to(DEVICE), y.to(DEVICE)
    return x, y

@torch.no_grad()
def estimate_loss(model, train_data, val_data):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(EVAL_ITERS)
        for k in range(EVAL_ITERS):
            X, Y = get_batch(split, train_data, val_data)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

def get_lr(it):
    if it < WARMUP_ITERS:
        return LEARNING_RATE * it / WARMUP_ITERS
    if it > LR_DECAY_ITERS:
        return MIN_LR
    decay_ratio = (it - WARMUP_ITERS) / (LR_DECAY_ITERS - WARMUP_ITERS)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return MIN_LR + coeff * (LEARNING_RATE - MIN_LR)

# defining the model

class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(N_EMBD, head_size, bias=False)
        self.query = nn.Linear(N_EMBD, head_size, bias=False)
        self.value = nn.Linear(N_EMBD, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(BLOCK_SIZE, BLOCK_SIZE)))
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (C // N_HEAD)**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for h in range(num_heads)])
        self.proj = nn.Linear(N_EMBD, N_EMBD)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, N_EMBD)
        self.position_embedding_table = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.blocks = nn.Sequential(*[Block(N_EMBD, n_head=N_HEAD) for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -BLOCK_SIZE:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Train or generate text with a Language Model.")
    parser.add_argument('--mode', choices=['train', 'generate'], required=True)
    parser.add_argument('--weights_path', type=str, default='model_weights.pth')
    parser.add_argument('--max_new_tokens', type=int, default=500)
    parser.add_argument('--prompt', type=str, default='\n')
    args = parser.parse_args(['--mode', 'train'])

    model = BigramLanguageModel()
    m = model.to(DEVICE)

    if args.mode == 'train':
        print(f"Using device: {DEVICE}")

        data = torch.tensor(encode(text), dtype=torch.long)
        n = int(0.9 * len(data))
        train_data = data[:n]
        val_data = data[n:]

        print(f"{sum(p.numel() for p in m.parameters())/1e6:.2f}M parameters")

        optimizer = torch.optim.AdamW(m.parameters(), lr=LEARNING_RATE)

        print("\nStarting training...")
        for iter in range(MAX_ITERS):
            lr = get_lr(iter)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

            if iter % EVAL_INTERVAL == 0 or iter == MAX_ITERS - 1:
                losses = estimate_loss(m, train_data, val_data)
                print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}, lr {lr:.6f}")

            xb, yb = get_batch('train', train_data, val_data)
            logits, loss = m(xb, yb)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP)
            optimizer.step()

        print("Training complete.")

        print(f"Saving model weights to {args.weights_path}...")
        torch.save(m.state_dict(), args.weights_path)
        print("Save complete.")

    elif args.mode == 'generate':
        print(f"Using device: {DEVICE}")

        print(f"Loading model weights from {args.weights_path}...")
        try:
            m.load_state_dict(torch.load(args.weights_path, map_location=DEVICE))
        except FileNotFoundError:
            print(f"Error: Weights file not found at '{args.weights_path}'. Please train the model first.")
            exit()
        except Exception as e:
            print(f"Error loading model weights: {e}")
            exit()

        m.eval()
        print("Model loaded successfully.")

        print("\nGenerating text...")
        start_context = torch.tensor(encode(args.prompt), dtype=torch.long, device=DEVICE).unsqueeze(0)
        generated_tokens = m.generate(start_context, max_new_tokens=args.max_new_tokens)[0].tolist()
        print(decode(generated_tokens))

Using device: mps
162.47M parameters

Starting training...
step 0: train loss 11.0044, val loss 11.0067, lr 0.000000


RuntimeError: MPS backend out of memory (MPS allocated: 17.91 GiB, other allocations: 1.71 GiB, max allowed: 20.13 GiB). Tried to allocate 1.53 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).